# Real Dataset Pipeline Notebook

Notebook ini khusus untuk menjalankan pipeline pada dataset asli (bukan sintetis).

Alur utama:
1. Siapkan/download dataset asli (Kaggle/URL/arsip lokal).
2. Siapkan annotation (langsung pakai file existing, atau auto-build untuk struktur folder CAER-S).
3. Jalankan Stage-1 pseudo-label generation dengan VLM lokal.
4. Jalankan Stage-2 training, evaluation, ablation opsional, attention visualization opsional.
5. Simpan laporan akhir dalam file markdown.

Catatan penting:
- Dataset seperti EMOTIC biasanya perlu lisensi/approval sehingga tidak selalu bisa diunduh bebas tanpa kredensial.
- Untuk download otomatis, set environment variable yang dibutuhkan sebelum menjalankan sel.

In [1]:
!nvidia-smi

Thu Apr  9 14:42:54 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.141.03   Driver Version: 470.141.03   CUDA Version: 11.6     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-SXM...  On   | 00000000:07:00.0 Off |                    0 |
| N/A   47C    P0   182W / 400W |  49347MiB / 81251MiB |     64%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

## Server Bootstrap (Set Sebelum Cell Setup)

Jika kernel Jupyter tidak membuka folder repo ini secara langsung, isi salah satu opsi berikut di cell berikutnya:
- Set `EMOTION_REPO_ROOT` ke path repo di server.
- Atau set `EMOTION_REPO_GIT_URL` agar notebook clone repo otomatis.

In [7]:
import os
from pathlib import Path
from typing import Iterable

def iter_subdirs_limited(root: Path, max_depth: int = 4, max_dirs: int = 5000) -> Iterable[Path]:
    skip_names = {".git", "__pycache__", ".venv", "venv", "node_modules", ".cache"}
    stack: list[tuple[Path, int]] = [(root, 0)]
    visited = 0
    while stack:
        current, depth = stack.pop()
        if not current.is_dir():
            continue
        visited += 1
        if visited > max_dirs:
            return
        yield current
        if depth >= max_depth:
            continue
        try:
            children = list(current.iterdir())
        except (PermissionError, OSError):
            continue
        for child in children:
            if child.is_dir() and child.name.lower() not in skip_names:
                stack.append((child, depth + 1))

def detect_repo_candidates() -> list[Path]:
    roots = [Path.home(), Path("/workspace"), Path("/mnt/workspace"), Path("/kaggle/working")]
    candidates: list[Path] = []
    seen: set[str] = set()
    for root in roots:
        if not root.exists():
            continue
        for directory in iter_subdirs_limited(root, max_depth=4, max_dirs=8000):
            if (directory / "scripts" / "train.py").exists() and (directory / "src" / "emotion_reasoning").exists():
                key = str(directory.resolve())
                if key not in seen:
                    seen.add(key)
                    candidates.append(directory.resolve())
    return candidates

# Opsi 1: gunakan path repo yang sudah ada di server
os.environ.setdefault("EMOTION_REPO_ROOT", "")  # contoh: /home/username/VLM-Research

# Opsi 2: clone otomatis jika repo belum ada di server
os.environ.setdefault("EMOTION_REPO_GIT_URL", "")  # contoh: https://github.com/owner/repo.git
os.environ.setdefault("EMOTION_REPO_BASE_DIR", "/home/taqiyudinmiftah")
os.environ.setdefault("EMOTION_REPO_DIRNAME", "VLM-Research")

if not os.environ.get("EMOTION_REPO_ROOT", "").strip():
    detected = detect_repo_candidates()
    if detected:
        os.environ["EMOTION_REPO_ROOT"] = str(detected[0])
        print(f"Auto-detected EMOTION_REPO_ROOT: {detected[0]}")
    else:
        print("No repo candidate auto-detected in common server locations.")

print("EMOTION_REPO_ROOT:", os.environ.get("EMOTION_REPO_ROOT") or "<not set>")
print("EMOTION_REPO_GIT_URL:", os.environ.get("EMOTION_REPO_GIT_URL") or "<not set>")

No repo candidate auto-detected in common server locations.
EMOTION_REPO_ROOT: <not set>
EMOTION_REPO_GIT_URL: <not set>


In [5]:
from __future__ import annotations

import csv
import json
import os
import random
import shlex
import subprocess
import sys
import tarfile
import urllib.request
import zipfile
from datetime import datetime
from pathlib import Path
from typing import Iterable


def is_repo_root(path: Path) -> bool:
    try:
        return (path / "scripts" / "train.py").exists() and (path / "src" / "emotion_reasoning").exists()
    except (PermissionError, OSError):
        return False


def unique_paths(paths: Iterable[Path]) -> list[Path]:
    output: list[Path] = []
    seen: set[str] = set()
    for path in paths:
        try:
            resolved = path.expanduser().resolve()
        except Exception:
            continue
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        output.append(resolved)
    return output


def iter_subdirs_limited(root: Path, max_depth: int = 4, max_dirs: int = 8000) -> Iterable[Path]:
    skip_names = {".git", "__pycache__", ".venv", "venv", "node_modules", ".cache"}
    stack: list[tuple[Path, int]] = [(root, 0)]
    visited = 0

    while stack:
        current, depth = stack.pop()
        if not current.is_dir():
            continue

        visited += 1
        if visited > max_dirs:
            return

        yield current

        if depth >= max_depth:
            continue

        try:
            children = list(current.iterdir())
        except (PermissionError, OSError):
            continue

        for child in children:
            if child.is_dir() and child.name.lower() not in skip_names:
                stack.append((child, depth + 1))


def build_search_roots() -> list[Path]:
    env_candidates = []
    for env_name in ("EMOTION_REPO_ROOT", "REPO_ROOT", "PROJECT_ROOT"):
        value = os.environ.get(env_name, "").strip()
        if value:
            env_candidates.append(Path(value))

    cwd = Path.cwd().resolve()
    home = Path.home().resolve()

    base_candidates = [
        cwd,
        *cwd.parents,
        home,
        home / "work",
        home / "workspace",
        Path("/workspace"),
        Path("/mnt/workspace"),
        Path("/kaggle/working"),
    ]

    return unique_paths([*env_candidates, *base_candidates])


def try_clone_repo_from_env() -> Path | None:
    repo_url = os.environ.get("EMOTION_REPO_GIT_URL", "").strip()
    if not repo_url:
        return None

    base_dir = Path(os.environ.get("EMOTION_REPO_BASE_DIR", str(Path.home()))).expanduser().resolve()
    repo_dirname = os.environ.get("EMOTION_REPO_DIRNAME", "VLM-Research").strip() or "VLM-Research"
    target = base_dir / repo_dirname

    if is_repo_root(target):
        return target

    base_dir.mkdir(parents=True, exist_ok=True)
    clone_cmd = ["git", "clone", "--depth", "1", repo_url, str(target)]
    print("$", " ".join(shlex.quote(arg) for arg in clone_cmd))
    try:
        subprocess.run(clone_cmd, check=True, text=True)
    except FileNotFoundError as exc:
        raise FileNotFoundError("git tidak tersedia pada kernel server. Install git atau set EMOTION_REPO_ROOT.") from exc

    return target if is_repo_root(target) else None


def find_project_root() -> Path:
    search_roots = build_search_roots()

    for candidate in search_roots:
        if is_repo_root(candidate):
            return candidate

    for root in search_roots:
        if not root.exists() or not root.is_dir():
            continue
        for candidate in iter_subdirs_limited(root, max_depth=4, max_dirs=8000):
            if is_repo_root(candidate):
                return candidate

    cloned_root = try_clone_repo_from_env()
    if cloned_root is not None:
        return cloned_root

    checked_roots = [str(path) for path in search_roots if path.exists()]
    roots_text = "\n".join(f"- {root}" for root in checked_roots) if checked_roots else "- <no existing roots checked>"
    raise FileNotFoundError(
        "Repo root tidak ditemukan pada kernel Jupyter server.\n"
        "Set env EMOTION_REPO_ROOT ke direktori repo yang berisi scripts/train.py dan src/emotion_reasoning.\n"
        "Atau set EMOTION_REPO_GIT_URL agar notebook melakukan clone otomatis.\n"
        "Contoh: export EMOTION_REPO_ROOT=/home/username/VLM-Research\n"
        f"Search roots checked:\n{roots_text}"
    )


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)


def run_cmd(args: list[str], capture: bool = False) -> subprocess.CompletedProcess:
    print("$", " ".join(shlex.quote(arg) for arg in args))
    return subprocess.run(
        args,
        cwd=str(PROJECT_ROOT),
        check=True,
        text=True,
        capture_output=capture,
    )


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def parse_json_from_stdout(stdout_text: str) -> dict:
    start = stdout_text.find("{")
    end = stdout_text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("No JSON object found in command output")
    return json.loads(stdout_text[start : end + 1])


SEED = 42
random.seed(SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")

FileNotFoundError: Repo root tidak ditemukan pada kernel Jupyter server.
Set env EMOTION_REPO_ROOT ke direktori repo yang berisi scripts/train.py dan src/emotion_reasoning.
Atau set EMOTION_REPO_GIT_URL agar notebook melakukan clone otomatis.
Contoh: export EMOTION_REPO_ROOT=/home/username/VLM-Research
Search roots checked:
- /home/taqiyudinmiftah
- /home
- /

## User Configuration (Jupyter Server)

Atur variabel berikut sebelum menjalankan pipeline.

Wajib diperhatikan untuk mode server:
- Jika kernel tidak start dari folder repo, set env `EMOTION_REPO_ROOT` ke direktori repo yang berisi `scripts/train.py` dan `src/emotion_reasoning`.

Pilihan sumber dataset asli:
- Kaggle slug: set env `CAERS_KAGGLE_SLUG` (format `owner/dataset-name`)
- URL arsip langsung: set env `REAL_DATA_ARCHIVE_URL`
- Arsip lokal: set env `REAL_DATA_ARCHIVE_PATH`
- Folder existing (tanpa download): set env `REAL_DATA_IMAGE_ROOT`

Kebutuhan Kaggle:
- Simpan API key di `~/.kaggle/kaggle.json` atau atur `KAGGLE_CONFIG_DIR`

In [ ]:
DATASET_NAME = "caer-s"  # pilihan: caer-s, emotic

USE_AUTO_DOWNLOAD = False
DOWNLOAD_METHOD = "none"  # pilihan: kaggle, url, local_archive, none

DATA_ROOT = PROJECT_ROOT / "data"
REAL_DATA_DIR = DATA_ROOT / "real_datasets"
WORK_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "real_dataset_runs"
WORK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional env-driven inputs
KAGGLE_DATASET_SLUG = os.environ.get("CAERS_KAGGLE_SLUG", "").strip()
ARCHIVE_URL = os.environ.get("REAL_DATA_ARCHIVE_URL", "").strip()
LOCAL_ARCHIVE_PATH = os.environ.get("REAL_DATA_ARCHIVE_PATH", "").strip()
USER_IMAGE_ROOT = os.environ.get("REAL_DATA_IMAGE_ROOT", "").strip()
USER_ANNOTATION_PATH = os.environ.get("REAL_DATA_ANNOTATION_PATH", "").strip()

# Stage-1 options
RUN_STAGE1 = True
VLM_TYPE = "moondream"  # choices: llava, moondream
VLM_MODEL = "vikhyatk/moondream2"
MAX_NEW_TOKENS = 128

# Stage-2 / analysis options
RUN_ABLATION = False
RUN_ATTENTION_VIZ = True
EPOCHS_OVERRIDE = 10
BATCH_SIZE_OVERRIDE = 8

print("Configuration loaded.")
print(f"DATASET_NAME={DATASET_NAME}, DOWNLOAD_METHOD={DOWNLOAD_METHOD}, USE_AUTO_DOWNLOAD={USE_AUTO_DOWNLOAD}")

In [ ]:
def ensure_kaggle_cli() -> None:
    try:
        run_cmd([sys.executable, "-m", "kaggle", "--help"])
    except Exception:
        run_cmd([sys.executable, "-m", "pip", "install", "--user", "kaggle"])


def resolve_kaggle_credentials_file() -> Path | None:
    candidates: list[Path] = []
    config_dir = os.environ.get("KAGGLE_CONFIG_DIR", "").strip()
    if config_dir:
        candidates.append(Path(config_dir).expanduser() / "kaggle.json")
    candidates.append(Path.home() / ".kaggle" / "kaggle.json")

    for path in candidates:
        if path.exists():
            return path
    return None


def ensure_kaggle_credentials() -> None:
    cred_path = resolve_kaggle_credentials_file()
    if cred_path is None:
        raise FileNotFoundError(
            "Kaggle API key tidak ditemukan. Simpan kaggle.json di ~/.kaggle/kaggle.json "
            "atau set KAGGLE_CONFIG_DIR terlebih dahulu."
        )
    print(f"Using Kaggle credentials at: {cred_path}")


def download_from_kaggle(slug: str, download_dir: Path) -> Path:
    if not slug:
        raise ValueError("Kaggle slug kosong. Set CAERS_KAGGLE_SLUG dahulu.")
    ensure_kaggle_credentials()
    ensure_kaggle_cli()
    download_dir.mkdir(parents=True, exist_ok=True)
    run_cmd([
        sys.executable,
        "-m",
        "kaggle",
        "datasets",
        "download",
        "-d",
        slug,
        "-p",
        str(download_dir),
        "--force",
    ])
    zip_files = sorted(download_dir.glob("*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not zip_files:
        raise FileNotFoundError(f"No zip archive produced by Kaggle for slug: {slug}")
    return zip_files[0]


def download_from_url(url: str, output_path: Path) -> Path:
    if not url:
        raise ValueError("Archive URL kosong. Set REAL_DATA_ARCHIVE_URL dahulu.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, output_path)
    return output_path


def extract_archive(archive_path: Path, output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    lower_name = archive_path.name.lower()
    if lower_name.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(output_dir)
    elif lower_name.endswith((".tar", ".tar.gz", ".tgz", ".tar.bz2", ".tbz2", ".tar.xz", ".txz")) or tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path, "r:*") as tf:
            tf.extractall(output_dir)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")
    return output_dir


def has_any_image_file(root: Path) -> bool:
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in image_exts:
            return True
    return False


def resolve_extracted_image_root(extracted_root: Path) -> Path:
    if has_any_image_file(extracted_root):
        return extracted_root

    for child in sorted(extracted_root.iterdir()):
        if child.is_dir() and has_any_image_file(child):
            return child

    return extracted_root


def resolve_raw_image_root() -> Path:
    if USER_IMAGE_ROOT:
        path = Path(USER_IMAGE_ROOT).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"USER_IMAGE_ROOT not found: {path}")
        return path

    dataset_root = (REAL_DATA_DIR / DATASET_NAME).resolve()
    dataset_root.mkdir(parents=True, exist_ok=True)

    if not USE_AUTO_DOWNLOAD or DOWNLOAD_METHOD == "none":
        if not has_any_image_file(dataset_root):
            raise FileNotFoundError(
                f"Tidak ada image file di {dataset_root}. "
                "Set REAL_DATA_IMAGE_ROOT atau aktifkan USE_AUTO_DOWNLOAD dengan metode yang sesuai."
            )
        return dataset_root

    archive_dir = dataset_root / "archives"
    extracted_dir = dataset_root / "extracted"

    if DOWNLOAD_METHOD == "kaggle":
        archive_path = download_from_kaggle(KAGGLE_DATASET_SLUG, archive_dir)
    elif DOWNLOAD_METHOD == "url":
        archive_path = download_from_url(ARCHIVE_URL, archive_dir / "dataset_archive.zip")
    elif DOWNLOAD_METHOD == "local_archive":
        if not LOCAL_ARCHIVE_PATH:
            raise ValueError("REAL_DATA_ARCHIVE_PATH kosong untuk mode local_archive")
        archive_path = Path(LOCAL_ARCHIVE_PATH).expanduser().resolve()
        if not archive_path.exists():
            raise FileNotFoundError(f"Local archive not found: {archive_path}")
    else:
        raise ValueError(f"Unknown DOWNLOAD_METHOD: {DOWNLOAD_METHOD}")

    extracted_root = extract_archive(archive_path=archive_path, output_dir=extracted_dir)
    return resolve_extracted_image_root(extracted_root)


RAW_IMAGE_ROOT = resolve_raw_image_root()
print(f"RAW_IMAGE_ROOT: {RAW_IMAGE_ROOT}")

In [ ]:
CAER_S_CLASSES = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad", "Surprise"]

def choose_class_names(dataset_name: str) -> list[str]:
    normalized = dataset_name.strip().lower()
    if normalized in {"caer-s", "caers"}:
        return CAER_S_CLASSES
    if normalized == "emotic":
        return [
            "Affection", "Anger", "Annoyance", "Anticipation", "Aversion",
            "Confidence", "Disapproval", "Disconnection", "Disquietment", "Doubt/Confusion",
            "Embarrassment", "Engagement", "Esteem", "Excitement", "Fatigue",
            "Fear", "Happiness", "Pain", "Peace", "Pleasure",
            "Sadness", "Sensitivity", "Suffering", "Surprise", "Sympathy", "Yearning"
        ]
    raise ValueError(f"Unsupported dataset: {dataset_name}")

def load_json_like(path: Path) -> list[dict]:
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        records = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return records
    if suffix == ".json":
        payload = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(payload, list):
            return payload
        if isinstance(payload, dict) and "records" in payload:
            return payload["records"]
        raise ValueError(f"Unsupported JSON structure in {path}")
    if suffix == ".csv":
        with path.open("r", encoding="utf-8", newline="") as f:
            return list(csv.DictReader(f))
    raise ValueError(f"Unsupported annotation extension: {path.suffix}")

def discover_and_build_caers_annotation(image_root: Path, output_path: Path) -> Path:
    class_lookup = {c.lower(): c for c in CAER_S_CLASSES}
    split_alias = {
        "train": {"train", "training"},
        "val": {"val", "valid", "validation", "dev"},
        "test": {"test", "testing"}
    }
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    records = []
    for path in image_root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in image_exts:
            continue
        lower_parts = [part.lower() for part in path.parts]
        split = None
        for canonical, aliases in split_alias.items():
            if any(token in lower_parts for token in aliases):
                split = canonical
                break
        label = None
        for token in lower_parts:
            if token in class_lookup:
                label = class_lookup[token]
                break
        if split is None or label is None:
            continue

        rel_path = path.relative_to(image_root).as_posix()
        sample_id = rel_path.replace("/", "_")
        records.append({
            "sample_id": sample_id,
            "image_path": rel_path,
            "labels": label,
            "split": split
        })

    if not records:
        raise RuntimeError("Tidak ada record CAER-S yang berhasil diparse. Set REAL_DATA_ANNOTATION_PATH manual.")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    return output_path

IMAGE_ROOT = Path(USER_IMAGE_ROOT).expanduser().resolve() if USER_IMAGE_ROOT else RAW_IMAGE_ROOT

if USER_ANNOTATION_PATH:
    ANNOTATION_PATH = Path(USER_ANNOTATION_PATH).expanduser().resolve()
    if not ANNOTATION_PATH.exists():
        raise FileNotFoundError(f"Annotation path not found: {ANNOTATION_PATH}")
else:
    if DATASET_NAME.strip().lower() == "emotic":
        raise RuntimeError(
            "EMOTIC biasanya membutuhkan file annotation resmi. Set REAL_DATA_ANNOTATION_PATH ke file json/jsonl/csv Anda."
        )
    generated_dir = DATA_ROOT / "prepared_real_annotations"
    ANNOTATION_PATH = discover_and_build_caers_annotation(
        image_root=IMAGE_ROOT,
        output_path=generated_dir / "caers_real_auto.jsonl"
    )

records_preview = load_json_like(ANNOTATION_PATH)
print(f"IMAGE_ROOT: {IMAGE_ROOT}")
print(f"ANNOTATION_PATH: {ANNOTATION_PATH}")
print(f"Total records loaded: {len(records_preview)}")
print("Sample record:")
print(records_preview[0] if records_preview else "No records")

In [ ]:
ANNOTATION_WITH_TEXT_PATH = ANNOTATION_PATH

if RUN_STAGE1:
    STAGE1_OUTPUT_PATH = WORK_OUTPUT_DIR / f"{DATASET_NAME}_pseudo_labels_real.jsonl"
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "generate_pseudo_labels.py"),
        "--annotation-path",
        str(ANNOTATION_PATH),
        "--image-root",
        str(IMAGE_ROOT),
        "--output-path",
        str(STAGE1_OUTPUT_PATH),
        "--dataset-name",
        DATASET_NAME,
        "--vlm-type",
        VLM_TYPE,
        "--vlm-model",
        VLM_MODEL,
        "--max-new-tokens",
        str(MAX_NEW_TOKENS)
    ]
    run_cmd(cmd)
    ANNOTATION_WITH_TEXT_PATH = STAGE1_OUTPUT_PATH
    print(f"Stage-1 output: {ANNOTATION_WITH_TEXT_PATH}")
else:
    print("RUN_STAGE1=False, memakai annotation input apa adanya.")

In [ ]:
BASE_CONFIG_MAP = {
    "caer-s": PROJECT_ROOT / "configs" / "caers_single_label.json",
    "caers": PROJECT_ROOT / "configs" / "caers_single_label.json",
    "emotic": PROJECT_ROOT / "configs" / "emotic_multimodal.json"
}

normalized_name = DATASET_NAME.strip().lower()
if normalized_name not in BASE_CONFIG_MAP:
    raise ValueError(f"Unsupported DATASET_NAME for runtime config: {DATASET_NAME}")

base_config_path = BASE_CONFIG_MAP[normalized_name]
runtime_config = json.loads(base_config_path.read_text(encoding="utf-8"))

experiment_name = f"{normalized_name}_real_multimodal"
run_output_dir = WORK_OUTPUT_DIR / experiment_name

runtime_config["experiment_name"] = experiment_name
runtime_config["dataset"]["name"] = normalized_name
runtime_config["dataset"]["annotation_path"] = str(ANNOTATION_WITH_TEXT_PATH)
runtime_config["dataset"]["image_root"] = str(IMAGE_ROOT)
runtime_config["training"]["output_dir"] = str(run_output_dir)

if EPOCHS_OVERRIDE is not None:
    runtime_config["training"]["epochs"] = int(EPOCHS_OVERRIDE)
if BATCH_SIZE_OVERRIDE is not None:
    runtime_config["training"]["batch_size"] = int(BATCH_SIZE_OVERRIDE)

RUNTIME_CONFIG_PATH = PROJECT_ROOT / "configs" / f"{experiment_name}.runtime.json"
write_json(RUNTIME_CONFIG_PATH, runtime_config)

print(f"Runtime config saved: {RUNTIME_CONFIG_PATH}")
print(json.dumps(runtime_config["dataset"], indent=2))

In [ ]:
train_cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "train.py"),
    "--config",
    str(RUNTIME_CONFIG_PATH)
]
run_cmd(train_cmd)

BEST_CHECKPOINT_PATH = Path(runtime_config["training"]["output_dir"]) / "best.pt"
if not BEST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"best.pt tidak ditemukan di: {BEST_CHECKPOINT_PATH}")

print(f"Training selesai. Best checkpoint: {BEST_CHECKPOINT_PATH}")

In [ ]:
eval_cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "evaluate.py"),
    "--config",
    str(RUNTIME_CONFIG_PATH),
    "--checkpoint",
    str(BEST_CHECKPOINT_PATH),
    "--split",
    "test"
]

eval_proc = run_cmd(eval_cmd, capture=True)
EVAL_PAYLOAD = parse_json_from_stdout(eval_proc.stdout)
EVAL_PATH = Path(runtime_config["training"]["output_dir"]) / "eval_test.json"
write_json(EVAL_PATH, EVAL_PAYLOAD)

print("Evaluation payload:")
print(json.dumps(EVAL_PAYLOAD, indent=2))
print(f"Saved: {EVAL_PATH}")

In [ ]:
ABLATION_PAYLOAD = None
ATTENTION_IMAGE_PATH = None

if RUN_ABLATION:
    ablation_cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "run_ablation.py"),
        "--config",
        str(RUNTIME_CONFIG_PATH)
    ]
    ablation_proc = run_cmd(ablation_cmd, capture=True)
    ABLATION_PAYLOAD = parse_json_from_stdout(ablation_proc.stdout)
    ablation_path = Path(runtime_config["training"]["output_dir"]) / "ablation_summary.json"
    write_json(ablation_path, ABLATION_PAYLOAD)
    print(f"Ablation summary saved: {ablation_path}")
else:
    print("RUN_ABLATION=False, skip ablation.")

if RUN_ATTENTION_VIZ:
    ATTENTION_IMAGE_PATH = Path(runtime_config["training"]["output_dir"]) / "attention_overlay.png"
    viz_cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "visualize_attention.py"),
        "--config",
        str(RUNTIME_CONFIG_PATH),
        "--checkpoint",
        str(BEST_CHECKPOINT_PATH),
        "--split",
        "test",
        "--sample-index",
        "0",
        "--output-path",
        str(ATTENTION_IMAGE_PATH)
    ]
    run_cmd(viz_cmd)
    print(f"Attention visualization saved: {ATTENTION_IMAGE_PATH}")
else:
    print("RUN_ATTENTION_VIZ=False, skip attention visualization.")

In [ ]:
report_lines = []
report_lines.append("# Real Dataset Experiment Report")
report_lines.append("")
report_lines.append(f"- Generated at: {datetime.utcnow().isoformat()}Z")
report_lines.append(f"- Dataset: {DATASET_NAME}")
report_lines.append(f"- Image root: {IMAGE_ROOT}")
report_lines.append(f"- Annotation input: {ANNOTATION_PATH}")
report_lines.append(f"- Stage-1 output: {ANNOTATION_WITH_TEXT_PATH}")
report_lines.append(f"- Runtime config: {RUNTIME_CONFIG_PATH}")
report_lines.append(f"- Checkpoint: {BEST_CHECKPOINT_PATH}")
report_lines.append("")

if "EVAL_PAYLOAD" in globals():
    metrics = EVAL_PAYLOAD.get("metrics", {})
    report_lines.append("## Test Metrics")
    report_lines.append("")
    report_lines.append(f"- mAP: {metrics.get('mAP')}")
    report_lines.append(f"- auc_roc: {metrics.get('auc_roc')}")
    report_lines.append(f"- accuracy: {metrics.get('accuracy')}")
    report_lines.append(f"- loss: {metrics.get('loss')}")
    report_lines.append("")

if ABLATION_PAYLOAD is not None:
    report_lines.append("## Ablation")
    report_lines.append("")
    for mode, payload in ABLATION_PAYLOAD.items():
        if not isinstance(payload, dict):
            continue
        best_val = payload.get("best_val_metrics", {})
        report_lines.append(f"- {mode}: mAP={best_val.get('mAP')}, acc={best_val.get('accuracy')}")
    report_lines.append("")

if ATTENTION_IMAGE_PATH is not None:
    report_lines.append("## Attention Visualization")
    report_lines.append("")
    report_lines.append(f"- File: {ATTENTION_IMAGE_PATH}")
    report_lines.append("")

REPORT_PATH = Path(runtime_config["training"]["output_dir"]) / "real_dataset_report.md"
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print(f"Report saved: {REPORT_PATH}")
print("\n".join(report_lines[:20]))